# Extract WASHC1 and WASHC2 monomers from best_complex.pdb

Chain mapping (from `chains.csv`):
- **Chain A** → WASHC1 (A8K0Z3)
- **Chain B** → WASHC2C (Q9Y4E1)

In [1]:
from pathlib import Path

INPUT_PDB  = Path(r"N:/08_NK_structure_prediction/data/WASH_complex/assembled_complex/output_interact_FKBP15_best/best_complex.pdb")
OUTPUT_DIR = INPUT_PDB.parent

CHAINS = {
    "A": "WASHC1",
    "B": "WASHC2",
}

In [2]:
def extract_chain(src: Path, chain_id: str, dst: Path) -> int:
    """Write ATOM/HETATM lines for one chain plus TER/END, return atom count."""
    count = 0
    with src.open() as fh_in, dst.open("w") as fh_out:
        for line in fh_in:
            record = line[:6].strip()
            if record in ("ATOM", "HETATM"):
                if line[21] == chain_id:
                    fh_out.write(line)
                    count += 1
        fh_out.write("TER\n")
        fh_out.write("END\n")
    return count


for chain_id, name in CHAINS.items():
    out_path = OUTPUT_DIR / f"{name}_monomer.pdb"
    n_atoms  = extract_chain(INPUT_PDB, chain_id, out_path)
    print(f"Chain {chain_id} ({name}): {n_atoms} atoms → {out_path.name}")

Chain A (WASHC1): 3545 atoms → WASHC1_monomer.pdb
Chain B (WASHC2): 10200 atoms → WASHC2_monomer.pdb


In [3]:
# Quick sanity check — residue count per monomer
for name in CHAINS.values():
    pdb = OUTPUT_DIR / f"{name}_monomer.pdb"
    resids = set()
    with pdb.open() as fh:
        for line in fh:
            if line[:4] in ("ATOM", "HETA"):
                resids.add((line[21], line[22:26].strip()))
    print(f"{name}: {len(resids)} residues")

WASHC1: 465 residues
WASHC2: 1320 residues
